In [ ]:
# we reload the packages as we make changes frequently
%load_ext autoreload
%autoreload 2

In [30]:
from tensorly_custom.contrib.sparse.decomposition import non_negative_tucker
from utils import DATA_DIR, torch_or_pickle_load, select_gpu
import torch
import pickle
import tensorflow as tf
import tensorly_custom as tl
import time

In [31]:
select_gpu()
with open(DATA_DIR / "tensors/fineweb_old/vocabularies/1000.pkl", "rb") as f:
    vocab = pickle.load(f)
vocab_v = vocab["vocab_v"]
vocab_s = vocab["vocab_s"]
vocab_o = vocab["vocab_o"]
v2i = vocab["v2i"]
s2i = vocab["s2i"]
o2i = vocab["o2i"]


def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)


class TupleTensor:
    def __init__(self, path: str):
        self.tensor = _torch_or_tucker_load(path)

    def sparse_representation(self):
        # we return the sparse representation of the tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if not isinstance(self.tensor, tf.Tensor):
            self.tensor = tf.convert_to_tensor(self.tensor)
        return tf.sparse.from_dense(self.tensor)

    def tensor_to_sparse(self):
        self.tensor = self.sparse_representation()

    def decompose(self, non_negative=True, hals=True, rank=None,
                  n_iter_max=100, init="random", tol=1e-12, verbose=False):
        if rank is None:
            rank = [100, 100, 100]
        tic = time.time()

        # if not non_negative:
        #     decomp = tucker
        # elif hals:
        #     decomp = non_negative_tucker_hals
        # else:
        #     decomp = non_negative_tucker
        decomp = non_negative_tucker

        core, factors = decomp(self.tensor, rank=rank, n_iter_max=n_iter_max,
                               init=init, tol=tol, verbose=verbose)
        toc = time.time()
        print("Time taken for decomposition", toc - tic)
        return core, factors

Selected GPU 3 with 454.62 MB used memory.


In [32]:
tensor = TupleTensor(DATA_DIR / "tensors/fineweb_dutch_vectors_ids/populated/sii_100.pt")

In [33]:
from tensorly_custom.contrib.sparse.decomposition import non_negative_tucker as nnt_sparse
from tensorly_custom.contrib.sparse import tensor as sparse_tensorly
import numpy as np
import sparse

tl.set_backend("numpy")
sparse_tensor_tensorly = sparse_tensorly(tensor.tensor)

In [34]:
result = nnt_sparse(sparse_tensor_tensorly, init="random", rank=[100,100,100], n_iter_max=10, verbose=True, tol=1e-10)


Elapsed time: 0 minutes and 6.29425048828125e-05 seconds.
initialization
Elapsed time: 0 minutes and 0.0754246711730957 seconds.
normalization
Elapsed time: 0 minutes and 0.07799196243286133 seconds.
	starting mode 0 update
tucker_to_tensor time: 0.6272368431091309
		tucker to tensor
		Elapsed time: 0 minutes and 0.6274433135986328 seconds.
		unfold and transpose
		Elapsed time: 0 minutes and 0.030240535736083984 seconds.
		numerator computation (dot product)
		Elapsed time: 0 minutes and 0.018207788467407227 seconds.
		numerator computation (clip)
		Elapsed time: 0 minutes and 0.0005693435668945312 seconds.
		denominator computation (dot products)
		Elapsed time: 0 minutes and 0.24721765518188477 seconds.
		denominator computation (clip)
		Elapsed time: 0 minutes and 0.0008180141448974609 seconds.
	mode 0 update complete
	Elapsed time: 0 minutes and 0.9278454780578613 seconds.
	starting mode 1 update
tucker_to_tensor time: 0.5805583000183105
		tucker to tensor
		Elapsed time: 0 minut

# making the multi_mode_dot function available for gpu

In [81]:
import numpy as np
import pickle
import torch
import os
from typing import List, Tuple
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from utils import DATA_DIR
from sympy.core.tests.test_args import test_sympy__core__function__Application


def _to_np(x):
    # Accept NumPy arrays or torch tensors; return NumPy view/copy
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return x

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)


def _role_index(role: str) -> int:
    if role == "verb":
        return 0
    elif role == "subject":
        return 1
    elif role == "object":
        return 2
    else:
        raise ValueError("role must be one of {'verb','subject','object'}")


class TuckerDecomposition:
    """Encapsulating the tucker decomposition (core and factors) and the vocabulary,
    providing methods for scoring, slicing, visualisation, etc."""
    def __init__(self, core, factors: List[torch.Tensor], vocab: dict):
        self.core = core
        self.factors = factors
        self.vocab = vocab

    # --- Construction and loading ---
    @classmethod
    def load_from_disk(cls,
                       dataset: str="karrewiet_vectors_ids",
                       method: str="counting",
                       sparse: bool=False,
                       dims: int=750,
                       rank: int=100,
                       iterations: int=1000,
                       map_location: str="cpu",
                          ) -> "TuckerDecomposition":

        """Loads a precomputed tucker decomposition from disk.
            Args:
                dataset (str): name of the dataset
                method (str): method used to compute the decomposition
                    - one of "counting", "sc", "sii"
                dims (int): dimensionality of the original tensor modes (vocab size)
                rank (int): rank of the decomposition
                iterations (int): number of iterations used to compute the decomposition
                map_location (str): device to map the loaded tensors to
            Returns:
                ((core, factors), vocab)
                    core: torch.Tensor
                    factors: list[torch.Tensor]
                    vocab: dict with keys 'vocab_v','vocab_s','vocab_o','v2i','s2i','o2i'
        """
        if method not in {"counting", "sc", "sii"}:
            raise ValueError("method must be one of {'counting','sc','sii'}")
        base = os.path.join(DATA_DIR, "tensors", dataset)

        vocab_path = os.path.join(base, f"vocabularies/{dims}.pkl")
        decomp_path = os.path.join(base,f"{'sparse'*sparse}" , "decomposition", f"{method}_{dims}d_{rank}r_{iterations}i.{'pt' if not sparse else 'pkl'}")
        if not os.path.exists(vocab_path):
            raise FileNotFoundError(f"Missing vocab file: {vocab_path}")
        if not os.path.exists(decomp_path):
            raise FileNotFoundError(f"Missing decomposition file: {decomp_path}")
        # the vocab is here under f"vocabularies_[dims].pkl"
        # Load with torch (they were saved with torch.save)
        with open(vocab_path, "rb") as f:
            vocab = pickle.load(f)
        (core, factors) = _torch_or_tucker_load(decomp_path, map_location=map_location)

        return cls(core, factors, vocab)

    def fetch_latents(self, triple: Tuple[str, str, str]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Fetches the latent representations for a given (verb, subject, object) triple."""
        # triple = (verb_str, subj_str, obj_str)
        v_idx = self.vocab["v2i"][triple[0]]
        s_idx = self.vocab["s2i"][triple[1]]
        o_idx = self.vocab["o2i"][triple[2]]
        V, S, O = [ _to_np(F) for F in self.factors]     # shapes (DIMS,R)
        v = V[v_idx]                                     # (R,)
        s = S[s_idx]                                     # (R,)
        o = O[o_idx]                                     # (R,)
        return v, s, o

    def _core_np(self):
        return _to_np(self.core)

    # -- Sparsity methods ---
    def sparse_representation(self, sparse_type):
        # we return the sparse representation of the tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if sparse_type == "tensorflow":
            if not isinstance(self.core, tf.Tensor):
                core = tf.convert_to_tensor(self.core)
            else:
                core = self.core
            sparse_core = tf.sparse.from_dense(core)
            # we do the same for the factors
            sparse_factors = []
            for factor in self.factors:
                if not isinstance(factor, tf.Tensor):
                    factor = tf.convert_to_tensor(factor)
                sparse_factor = tf.sparse.from_dense(factor)
                sparse_factors.append(sparse_factor)

            return sparse_core, sparse_factors
        elif sparse_type == "torch":
            if not isinstance(self.core, torch.Tensor):
                core = torch.tensor(self.core)
            else:
                core = self.core
            sparse_core = core.to_sparse()
            # we do the same for the factors
            sparse_factors = []
            for factor in self.factors:
                if not isinstance(factor, torch.Tensor):
                    factor = torch.tensor(factor)
                sparse_factor = factor.to_sparse()
                sparse_factors.append(sparse_factor)
            return sparse_core, sparse_factors

        else:
            raise NotImplementedError(f"Sparse representation for type {sparse_type} not implemented.")



    def tensor_to_sparse(self, sparse_type="tensorflow"):
        self.core, self.factors = self.sparse_representation(sparse_type)


    def tensor_to_dense(self):
        self.core = self._core_np().todense()
        self.factors = [_to_np(factor).todense() for factor in self.factors]







In [83]:
small_tensor = TuckerDecomposition.load_from_disk(
    dataset="fineweb_old",
    method="sii",
    sparse=False,
    dims=500,
    rank=100,
    iterations=1000,
    map_location="cpu"
)
small_tensor.tensor_to_sparse("torch")

In [58]:
import tensorflow as tf
import numpy as np
import sparse

def tf_sparse_to_pydatasparse(st: tf.SparseTensor) -> sparse.COO:
    """Converts a TensorFlow SparseTensor to a PyData/Sparse COO tensor."""
    # In TF2 eager mode:
    coords = st.indices().numpy()       # shape (nnz, ndim)
    data   = st.values().numpy()        # shape (nnz,)
    shape  = tuple(st.size())  # e.g. (d0, d1, ..., d_{n-1})

    return sparse.COO(coords, data, shape=shape)
# Convert the TensorFlow SparseTensor to PyData/Sparse COO tensor
sparse_core = tf_sparse_to_pydatasparse(small_tensor.core)
sparse_factors = [tf_sparse_to_pydatasparse(factor) for factor in small_tensor.factors]

[[ 0  0  0 ... 99 99 99]
 [ 0  0  0 ... 99 99 99]
 [ 0  1  2 ... 27 29 84]]
[1.9767789e+02 1.2622920e+03 2.4130750e+03 ... 2.3050513e+02 1.4012985e-45
 1.2984012e+02]
(100, 100, 100)
[[  0   0   0 ... 499 499 499]
 [  0   1   2 ...  97  98  99]]
[2.802597e-45 2.802597e-45 5.211009e-41 ... 3.453291e-09 2.212982e-15
 1.401298e-45]
(500, 100)
[[  0   0   0 ... 499 499 499]
 [  0   1   2 ...  96  97  98]]
[2.0828426e+01 2.8025969e-45 2.3822074e-44 ... 3.5857138e-12 3.1704307e-02
 1.4012985e-45]
(500, 100)
[[  0   1   1 ... 499 499 499]
 [  1   0   1 ...  97  98  99]]
[1.5230409e+01 2.6506155e+01 1.9541107e-41 ... 3.1195877e-02 1.4012985e-45
 1.4012985e-45]
(500, 100)


In [59]:
from tensorly_custom.contrib.sparse.backend import tucker_to_tensor
from tensorly_custom.contrib.sparse.tenalg import multi_mode_dot
reconstructed = tucker_to_tensor((sparse_core, sparse_factors))

tucker_to_tensor time: 58.49751424789429


# Plan: implementing this on cuda!

In [61]:
sparse.dot(sparse_core, sparse_factors[1].T)

Format,coo
Data Type,float32
Shape,"(100, 100, 500)"
nnz,4892895
Density,0.978579
Read-only,True
Size,130.7M
Storage ratio,6.85


In [65]:
import cupy as cp
import cupyx
# Convert PyData/Sparse COO tensor to CuPy sparse matrix
def pydatasparse_to_cupy_sparse(ps: sparse.COO) -> cupyx.scipy.sparse.coo_matrix:
    """Converts a PyData/Sparse COO tensor to a CuPy sparse matrix."""
    coords = cp.array(ps.coords)  # shape (ndim, nnz)
    data   = cp.array(ps.data)    # shape (nnz,)
    shape  = tuple(ps.shape)      # e.g. (d0, d1, ..., d_{n-1})
    print(coords.shape, data.shape, shape)
    # we make sure that three-dimensional tensors are converted to 1D vectors
    if len(shape) == 3:
        # we reshape the coords and shape accordingly
        coords = cp.vstack([
            coords[0],
            coords[1] * shape[2] + coords[2]
        ])
        shape = (shape[0], shape[1] * shape[2])
    return cupyx.scipy.sparse.coo_matrix((data, coords), shape=shape)
# Convert the sparse core and factors to CuPy sparse matrices
cupy_sparse_core = pydatasparse_to_cupy_sparse(sparse_core)
cupy_sparse_factors = [pydatasparse_to_cupy_sparse(factor) for factor in sparse_factors]


(3, 227999) (227999,) (100, 100, 100)
(2, 37859) (37859,) (500, 100)
(2, 31881) (31881,) (500, 100)
(2, 42753) (42753,) (500, 100)


In [73]:
# we print the shapes to verify
print("Core shape:", cupy_sparse_core.shape)
for i, factor in enumerate(cupy_sparse_factors):
    print(f"Factor {i} shape:", factor.shape)
dense_factor = small_tensor.factors[0].to_dense()
# we need this to be a matrix
dense_factor = dense_factor.numpy()

Core shape: (100, 10000)
Factor 0 shape: (500, 100)
Factor 1 shape: (500, 100)
Factor 2 shape: (500, 100)


In [76]:
cupy_sparse_core    = pydatasparse_to_cupy_sparse(sparse_core)
cupy_sparse_factors = [pydatasparse_to_cupy_sparse(factor) for factor in sparse_factors]

factor0_sparse = cupy_sparse_factors[0]      # shape (500, 100)
A = factor0_sparse.toarray()                 # make it dense on GPU

Y_mode1 = A @ cupy_sparse_core               # (500 x 10000)
print(Y_mode1.shape)
Y = Y_mode1.toarray().reshape(500, 100, 100)

(3, 227999) (227999,) (100, 100, 100)
(2, 37859) (37859,) (500, 100)
(2, 31881) (31881,) (500, 100)
(2, 42753) (42753,) (500, 100)
(500, 10000)


AttributeError: 'ndarray' object has no attribute 'toarray'

In [78]:
import cupy as cp
import cupyx
from cupyx.scipy.sparse import coo_matrix
import sparse  # PyData/Sparse


# Convert PyData/Sparse COO tensor to CuPy sparse matrix
def pydatasparse_to_cupy_sparse(ps: sparse.COO) -> coo_matrix:
    """Converts a PyData/Sparse COO tensor to a CuPy sparse matrix.

    For 3D inputs of shape (I, J, K), this returns a 2D matrix of shape
    (I, J*K), corresponding to the mode-1 unfolding.
    """
    coords = cp.array(ps.coords)  # shape (ndim, nnz)
    data   = cp.array(ps.data)    # shape (nnz,)
    shape  = tuple(ps.shape)      # e.g. (d0, d1, ..., d_{n-1})

    # we make sure that three-dimensional tensors are converted to 2D matrices
    # (mode-1 unfolding): (I, J, K) -> (I, J*K)
    if len(shape) == 3:
        I, J, K = shape
        coords = cp.vstack([
            coords[0],               # i
            coords[1] * K + coords[2]  # j*K + k
        ])
        shape = (I, J * K)           # 100 x (100*100) = 100 x 10000

    return coo_matrix((data, coords), shape=shape)


# --- Example: core tensor (100 x 100 x 100) and factor (500 x 100) --- #

# sparse_core: PyData/Sparse COO with shape (100, 100, 100)
# factor:      PyData/Sparse COO or numpy/cupy array with shape (500, 100)

# 1. Convert the sparse core to a CuPy sparse matrix (mode-1 unfolding)
cupy_sparse_core = pydatasparse_to_cupy_sparse(sparse_core)
# cupy_sparse_core.shape == (100, 100 * 100) == (100, 10000)

I, J, K = sparse_core.shape  # (100, 100, 100)

# 2. Convert the factor to a CuPy *dense* matrix of shape (500, 100)
#    If your factor is PyData/Sparse COO:
# factor_ps: sparse.COO with shape (500, 100)
factor_ps = sparse_factors[0]  # or however you pick your factor
R, I_factor = factor_ps.shape
assert I_factor == I  # sanity check: inner dim must match core's first mode

# get a dense CuPy array for the factor
# (factor_ps.todense() -> numpy array; cp.asarray moves it to GPU)
A = cp.asarray(factor_ps.todense())  # shape (500, 100)

# 3. Perform the sparse–dense matmul:
#    (500 x 100) @ (100 x 10000) -> (500 x 10000)
Y_mode1 = A @ cupy_sparse_core
print("Y_mode1 shape:", Y_mode1.shape)  # should be (500, 10000)
# Result is a sparse matrix type; convert to dense since the product is generally dense
Y_mode1_dense = Y_mode1  # cupy.ndarray, shape (500, 10000)

# 4. Refold back to a 3D tensor: (500, 100, 100)
Y = Y_mode1_dense.reshape(R, J, K)  # shape (500, 100, 100)


Y_mode1 shape: (500, 10000)


In [94]:
Y.shape

(500, 100, 100)

In [89]:
# we check if this is the same as doing it with full dense tensors
from tensorly import tucker_to_tensor
import tensorly
tensorly.set_backend("pytorch")
small_tensor_dense = TuckerDecomposition.load_from_disk(
    dataset="fineweb_old",
    method="sii",
    sparse=False,
    dims=500,
    rank=100,
    iterations=1000,
    map_location="cpu"
)

In [118]:
# we perform the same operations on the dense counterparts
core_dense = small_tensor_dense.core      # shape (100, 100, 100)
factor0_dense = small_tensor_dense.factors[0]  # shape (500, 100)
print(core_dense.shape, factor0_dense.shape)
# we do the dot product
Y_mode1_dense_full = torch.matmul(factor0_dense, core_dense.reshape(100, 10000))
Y_mode1_dense_full = Y_mode1_dense_full.reshape(500, 100, 100)

torch.Size([100, 100, 100]) torch.Size([500, 100])


In [119]:
# we check if they are similar
# we convert both to the same format first
y_torch = torch.tensor(Y.get())  # cupy to torch
print(y_torch.shape, Y_mode1_dense_full.shape)
# we compute the difference
difference = np.abs(y_torch - Y_mode1_dense_full)

torch.Size([500, 100, 100]) torch.Size([500, 100, 100])


In [124]:
y_torch

tensor([[[3.6606e-04, 8.8967e-28, 4.7876e-14,  ..., 7.1474e-04,
          9.2379e-15, 1.2188e-02],
         [6.0787e-04, 9.4266e-25, 5.1792e-04,  ..., 2.9632e-04,
          5.5540e-03, 1.0141e-04],
         [7.7357e-25, 1.5801e-13, 8.2605e-04,  ..., 7.1002e-04,
          4.6532e-25, 1.3796e-21],
         ...,
         [1.0688e-28, 1.2751e-03, 2.0849e-38,  ..., 2.1230e-36,
          4.9902e-29, 1.2167e-32],
         [3.4809e-04, 5.7579e-24, 1.1807e-38,  ..., 1.7355e-40,
          4.8333e-22, 1.9070e-36],
         [5.2620e-04, 1.6816e-28, 1.2375e-03,  ..., 1.5733e-37,
          4.0904e-42, 8.9630e-40]],

        [[7.1235e-18, 9.6823e-21, 2.8661e-17,  ..., 4.1705e-12,
          8.8209e-22, 1.4602e-21],
         [1.5111e-03, 6.7739e-26, 1.4027e-18,  ..., 9.1297e-22,
          4.5704e-18, 1.2913e-21],
         [6.1137e-17, 4.3481e-21, 5.5841e-04,  ..., 8.6118e-18,
          7.2175e-04, 2.5712e-24],
         ...,
         [2.0740e-26, 1.1877e-03, 1.7435e-26,  ..., 2.4924e-23,
          2.542

In [122]:
Y_mode1_dense_full

tensor([[[3.6606e-04, 8.8967e-28, 4.7876e-14,  ..., 7.1474e-04,
          9.2379e-15, 1.2188e-02],
         [6.0787e-04, 9.4266e-25, 5.1792e-04,  ..., 2.9632e-04,
          5.5540e-03, 1.0141e-04],
         [7.7357e-25, 1.5801e-13, 8.2605e-04,  ..., 7.1002e-04,
          4.6532e-25, 1.3796e-21],
         ...,
         [1.0688e-28, 1.2751e-03, 2.0849e-38,  ..., 2.1230e-36,
          4.9902e-29, 1.2167e-32],
         [3.4809e-04, 5.7579e-24, 1.1807e-38,  ..., 1.7355e-40,
          4.8333e-22, 1.9070e-36],
         [5.2620e-04, 1.6816e-28, 1.2375e-03,  ..., 1.5733e-37,
          4.0904e-42, 8.9630e-40]],

        [[7.1235e-18, 9.6823e-21, 2.8661e-17,  ..., 4.1705e-12,
          8.8209e-22, 1.4602e-21],
         [1.5111e-03, 6.7739e-26, 1.4027e-18,  ..., 9.1297e-22,
          4.5704e-18, 1.2913e-21],
         [6.1137e-17, 4.3481e-21, 5.5841e-04,  ..., 8.6118e-18,
          7.2175e-04, 2.5712e-24],
         ...,
         [2.0740e-26, 1.1877e-03, 1.7435e-26,  ..., 2.4924e-23,
          2.542

In [125]:
# we print the largest differences
print("Max difference:", difference.max().item())

Max difference: 6.984919309616089e-10


In [126]:
# for comparison, the max value in each of the tensors
print("Max value in sparse-based Y:", y_torch.max().item())
print("Max value in dense-based Y:", Y_mode1_dense_full.max().item())

Max value in sparse-based Y: 0.05885610729455948
Max value in dense-based Y: 0.05885610729455948


In [129]:
from tensorly.tenalg import multi_mode_dot as mmd
core_dense = small_tensor_dense.core          # torch, (100, 100, 100)
factor0_dense = small_tensor_dense.factors[0] # torch, (500, 100)

reconstructed_dense_disk = mmd(core_dense, [factor0_dense], modes=[0])

Y_np = cp.asnumpy(Y)
y_torch = torch.from_numpy(Y_np).to(reconstructed_dense_disk.dtype)

diff_disk = (y_torch - reconstructed_dense_disk).abs()
print("max abs diff vs disk:", diff_disk.max().item())


max abs diff vs disk: 6.984919309616089e-10


In [133]:
import tensorly as tl
tl.set_backend("cupy")
core_cupy = tl.tensor(sparse_core.todense())          # cupy, (100, 100, 100)
factor0_cupy = tl.tensor(sparse_factors[0].todense()) #

In [134]:
# we perform the multi_mode_dot on cupy tensors
reconstructed_cupy = tl.tenalg.multi_mode_dot(core_cupy, [factor0_cupy], modes=[0])

In [138]:
# we check the difference with the previous result
Y_np = cp.asnumpy(Y)
y_cupy = tl.tensor(Y_np)
diff_cupy = (y_cupy - reconstructed_cupy)
diff_cupy

array([[[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  4.9303807e-32,  0.0000000e+00],
        ...,
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         -0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]],

       [[-8.2718061e-25,  0.0000000e+00, -3.3087225e-24, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
        [ 0.0000000e+00,  0.0000000e+00, -1.0339758e-25, ...,
          0.0000000e+00,  4.1359031e-25,  0.0000000e+00],
        [ 6.6174449e-24, 